In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/test_random_400-2.csv
/kaggle/input/test_top_cosine_200-2.csv
/kaggle/input/test_top_rougeL_200-2.csv
/kaggle/input/train_data.csv
/kaggle/input/test_random_600-2.csv


In [2]:
!pip install -q -U transformers accelerate datasets peft bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 91.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.8/375.8 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 556.4/556.4 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 82.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 89.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 69.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2

In [3]:
from huggingface_hub import login

login("your_token_here")

HTTPError: Invalid user token.

In [ ]:
# ============================================================
# ENV FIX – chống phân mảnh CUDA (Cách 4)
# ============================================================
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# ============================================================
# IMPORTS
# ============================================================
import gc
import random
import json
import torch
import pandas as pd
from peft import (
    LoraConfig, 
    get_peft_model, 
    prepare_model_for_kbit_training, 
    PeftModel
)
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
)
from torch.utils.data import Dataset
from torch.nn.utils.rnn import pad_sequence


# ============================================================
# Disable Flash Attention v2 cho Gemma (bắt buộc)
# ============================================================
if hasattr(torch.backends, "cuda"):
    torch.backends.cuda.enable_flash_sdp(False)
    torch.backends.cuda.enable_mem_efficient_sdp(False)
    torch.backends.cuda.enable_math_sdp(True)


# ============================================================
# PATHS
# ============================================================
TRAIN_CSV = "/kaggle/input/project-multi/train-1.csv"
TEST_CSV  = "/kaggle/input/project-multi/test_random_400-2.csv"

assert os.path.exists(TRAIN_CSV)
assert os.path.exists(TEST_CSV)


# ============================================================
# LOAD DATA
# ============================================================
train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)

INPUT_COL = "readme" if "readme" in train_df.columns else \
            ("description_html" if "description_html" in train_df.columns else None)
assert INPUT_COL is not None

TARGET_COL = "description"
assert TARGET_COL in train_df.columns


# ============================================================
# PROMPT FUNCTION
# ============================================================
def build_prompt(description_html: str) -> str:
    return (
        "You are an expert app store editor. "
        "Given the following app description in HTML format, summarize it in 2-3 sentences, "
        "with a concise, engaging short description (max 80 characters). "
        f"App Description HTML:\n{description_html}\n"
        "Format your response as:\n"
        "Short Description: <your short description>\n\n"
    )


# ============================================================
# BUILD TRAIN ITEMS
# ============================================================
records = []
for _, row in train_df.iterrows():
    html = str(row[INPUT_COL])
    target = str(row[TARGET_COL]).strip()
    records.append({"prompt": build_prompt(html), "response": target})

random.seed(42)
random.shuffle(records)

split_idx = int(0.9 * len(records))
train_items = records[:split_idx]
dev_items   = records[split_idx:] if split_idx < len(records) else records[:1]


# ============================================================
# DATASET
# ============================================================
class PromptDataset(Dataset):
    def __init__(self, items, tokenizer, max_len=2048):
        self.items = items
        self.tok = tokenizer
        self.max_len = max_len

    def __len__(self): return len(self.items)

    def __getitem__(self, idx):
        ex = self.items[idx]
        prompt_ids = self.tok(
            ex["prompt"],
            add_special_tokens=False,
            truncation=True,
            max_length=self.max_len
        )["input_ids"]

        response_ids = self.tok(
            ex["response"] + self.tok.eos_token,
            add_special_tokens=False,
            truncation=True,
            max_length=self.max_len
        )["input_ids"]

        input_ids = prompt_ids + response_ids
        labels = [-100]*len(prompt_ids) + response_ids

        # truncate right
        if len(input_ids) > self.max_len:
            input_ids = input_ids[-self.max_len:]
            labels    = labels[-self.max_len:]

        return {
            "input_ids": torch.tensor(input_ids),
            "labels": torch.tensor(labels),
            "attention_mask": torch.ones(len(input_ids)),
        }


# ============================================================
# COLLATE
# ============================================================
def collate_fn(batch):
    pad_id = tokenizer.pad_token_id
    input_ids = pad_sequence([b["input_ids"] for b in batch], batch_first=True, padding_value=pad_id)
    labels    = pad_sequence([b["labels"]    for b in batch], batch_first=True, padding_value=-100)
    attn      = pad_sequence([b["attention_mask"] for b in batch], batch_first=True, padding_value=0)
    return {"input_ids": input_ids, "labels": labels, "attention_mask": attn}


# ============================================================
# MODEL LOADING
# ============================================================
MODEL_NAME = "google/gemma-2-2b-it"

gpu_ok = torch.cuda.is_available()
bf16_ok = gpu_ok and torch.cuda.get_device_capability(0)[0] >= 8
compute_dtype = torch.bfloat16 if bf16_ok else torch.float16
device_map = {"": 0}

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    torch_dtype=compute_dtype,
    device_map=device_map,
    attn_implementation="eager",
)

base_model = prepare_model_for_kbit_training(base_model)

lora_cfg = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)

model = get_peft_model(base_model, lora_cfg)
model.config.use_cache = False


# ============================================================
# BUILD DATASETS
# ============================================================
train_ds = PromptDataset(train_items, tokenizer, max_len=2048)
dev_ds   = PromptDataset(dev_items,   tokenizer, max_len=2048)


# ============================================================
# TRAINING ARGS — (Cách 3 added)
# ============================================================
args = TrainingArguments(
    output_dir="./gemma2_adapter",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    num_train_epochs=3,
    learning_rate=1e-4,
    fp16=not bf16_ok,
    bf16=bf16_ok,
    logging_steps=10,
    save_strategy="epoch",
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},  # 🔥 CÁCH 3
    report_to="none",
)


# ============================================================
# TRAIN
# ============================================================
import time
start = time.time()

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=dev_ds,
    data_collator=collate_fn,
)

trainer.train()

print(f"\n=== Training Done: {(time.time()-start)/60:.2f} minutes ===")


# ============================================================
# SAVE MODEL
# ============================================================
model.save_pretrained("./gemma2_adapter")
tokenizer.save_pretrained("./gemma2_adapter")

del trainer, model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


# ============================================================
# LOAD FOR INFERENCE
# ============================================================
base_inf = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    torch_dtype=compute_dtype,
    device_map=device_map,
    attn_implementation="eager",
)
model_inf = PeftModel.from_pretrained(base_inf, "./gemma2_adapter")
model_inf.eval()


TEST_INPUT_COL = (
    "description_html_clean" if "description_html_clean" in test_df.columns else
    "description_html"
)


# ============================================================
# GENERATE FUNCTION
# ============================================================
@torch.inference_mode()
def generate_short(html: str, max_new_tokens=64):
    prompt = build_prompt(html)
    inp = tokenizer(prompt, return_tensors="pt").to(model_inf.device)
    out = model_inf.generate(
        **inp,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=0.0,
    )
    gen_tokens = out[0][inp["input_ids"].shape[1]:]
    return tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()


# ============================================================
# RUN MAIN TEST
# ============================================================
test_df["pred_short_description"] = [
    generate_short(str(x)) for x in test_df[TEST_INPUT_COL].astype(str)
]
test_df.to_csv("predictions.csv", index=False)
print("Saved predictions.csv")

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

train_texts = train_df[INPUT_COL].astype(str).tolist()
test_texts  = test_df[TEST_INPUT_COL].astype(str).tolist()


train_set = set(train_texts)
test_set  = set(test_texts)
exact_overlap = train_set.intersection(test_set)

print(f"Số lượng test samples: {len(test_texts)}")
print(f"Số lượng test trùng EXACT với train: {len(exact_overlap)}")
print(f"Tỉ lệ exact overlap: {len(exact_overlap)/len(test_texts):.2%}")


vectorizer = TfidfVectorizer(max_features=5000).fit(train_texts + test_texts)
train_vecs = vectorizer.transform(train_texts)
test_vecs  = vectorizer.transform(test_texts)

threshold = 0.8 
similar_count = 0
for i, test_vec in enumerate(test_vecs):
    sims = cosine_similarity(test_vec, train_vecs).flatten()
    if sims.max() >= threshold:
        similar_count += 1
